<a href="https://colab.research.google.com/github/meghansn/mental-health-llm-pipeline/blob/rag-disorders/Symptom_Redaction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#intiate
# =====================================================
# Symptom Extraction from Therapy Transcripts
# =====================================================

import pandas as pd
import json
import vertexai

from google.cloud import bigquery
from google.colab import auth
from vertexai.generative_models import GenerativeModel

# Authenticate
auth.authenticate_user()

# Project Configuration
PROJECT_ID = "mental-health-llm-pip"
LOCATION = "us-central1"

# Initialize Clients
client = bigquery.Client(project=PROJECT_ID)

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION
)

# Load Gemini
model = GenerativeModel("gemini-2.5-flash")

print("BigQuery connected")
print("Vertex AI initialized")
print("Gemini loaded")

In [ ]:
query = """
SELECT
    session_id,
    transcript_redacted
FROM `mental-health-llm-pip.patient_insights.fact_sessions_redacted`
LIMIT 5
"""

df = client.query(query).to_dataframe()

print(df.shape)
df.head()

In [ ]:
print(df.iloc[0]["transcript_redacted"])

In [ ]:
transcript = df.iloc[0]["transcript_redacted"]

print(transcript[:3000])

In [ ]:
#extraction prompt
prompt = f"""
Extract mental health symptoms from the therapy transcript.

Requirements:
- Return ONLY a JSON array
- Use concise symptom names
- Do not return diagnoses
- Do not return explanations
- Do not return markdown

Example:
["insomnia", "fatigue", "hopelessness"]

Transcript:
{transcript}
"""

In [ ]:
response = model.generate_content(prompt)

print(response.text)

In [ ]:
import json

symptoms = json.loads(response.text)

print(symptoms)
print(type(symptoms))

In [ ]:
results = []

for _, row in df.iterrows():

    prompt = f"""
    Extract mental health symptoms from the therapy transcript.

    Requirements:
    - Return ONLY a JSON array
    - Use concise symptom names
    - Do not return diagnoses
    - Do not return explanations
    - Do not return markdown

    Example:
    ["insomnia", "fatigue", "hopelessness"]

    Transcript:
    {row['transcript_redacted']}
    """

    response = model.generate_content(prompt)

    try:
        cleaned = (
            response.text
            .replace("```json", "")
            .replace("```", "")
            .strip()
        )

        symptoms = json.loads(cleaned)

    except json.JSONDecodeError:
        print(
            f"Warning: Could not decode JSON for session {row['session_id']}. "
            f"Response: {response.text}"
        )
        symptoms = []

    results.append({
        "session_id": row["session_id"],
        "symptoms": symptoms
    })

print(f"Processed {len(results)} sessions")
results

In [ ]:
symptoms_df = pd.DataFrame(results)

symptoms_df.head()

In [ ]:
for row in results:
    print("\nSESSION:", row["session_id"])
    print(row["symptoms"])

In [ ]:
symptoms_df = pd.DataFrame(results)

symptoms_df.head()

In [ ]:
import json
from google.cloud import bigquery

query = """
SELECT
    session_id,
    transcript_redacted
FROM `mental-health-llm-pip.patient_insights.fact_sessions_redacted`
WHERE symptoms IS NULL
   OR ARRAY_LENGTH(symptoms) = 0
"""

df = client.query(query).to_dataframe()

print(f"Found {len(df)} sessions to process")

for i, (_, row) in enumerate(df.iterrows(), start=1):

    prompt = f"""
Extract clinically relevant mental health symptoms from the therapy transcript.

Requirements:
- Return ONLY a JSON array
- Use concise symptom names
- Do not return diagnoses
- Do not return explanations
- Do not return markdown

Example:
["insomnia","fatigue","hopelessness"]

Transcript:
{row['transcript_redacted']}
"""

    try:
        response = model.generate_content(prompt)

        cleaned = (
            response.text
            .replace("```json", "")
            .replace("```", "")
            .strip()
        )

        symptoms = json.loads(cleaned)

        update_query = """
        UPDATE `mental-health-llm-pip.patient_insights.fact_sessions_redacted`
        SET symptoms = @symptoms
        WHERE session_id = @session_id
        """

        job_config = bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter(
                    "session_id",
                    "STRING",
                    row["session_id"]
                ),
                bigquery.ArrayQueryParameter(
                    "symptoms",
                    "STRING",
                    symptoms
                )
            ]
        )

        client.query(update_query, job_config=job_config).result()

        if i % 50 == 0:
            print(f"Processed {i} of {len(df)} sessions")

    except Exception as e:
        print(f"Error processing session {row['session_id']}: {e}")

print("Finished processing all sessions")